In [6]:

df = pd.read_csv("modeling_dataset.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())



Dataset loaded successfully!
Dataset shape: (1434, 16)


,demarcation.code,financial_year_end.year,total_revenue,total_expenditure,operating_surplus,cash_coverage_ratio,collection_rate,debtor_days,repairs_maintenance_ratio,irregular_expenditure_ratio,grant_dependency,prior_audit_outcome,prior_audit_opinion_raw,Name,target_next_year,target_opinion_raw
0,BUF,2018,5.504778e+09,6.105515e+09,-6.007370e+08,3.581674,1.000000,0.000000,0.000000,0.034939,0.000000,at_risk,qualified,Buffalo City,at_risk,qualified
1,BUF,2019,6.011582e+09,6.877780e+09,-8.661981e+08,2.044205,1.000000,0.000000,0.056371,0.019271,0.000000,at_risk,qualified,Buffalo City,at_risk,qualified
2,BUF,2020,6.668512e+09,7.830443e+09,-1.161931e+09,2.105185,1.000000,0.000000,0.000048,0.038585,1.813338,at_risk,qualified,Buffalo City,at_risk,qualified
3,BUF,2021,7.884807e+09,8.276988e+09,-3.921814e+08,1.660615,1.000000,469.724040,0.046620,0.037601,0.957037,at_risk,qualified,Buffalo City,not_at_risk,unqualified_emphasis_of_matter
4,BUF,2022,8.249186e+09,9.201096e+09,-9.519095e+08,0.909618,0.770887,587.671118,0.043625,0.708989,1.097730,not_at_risk,unqualified_emphasis_of_matter,Buffalo City,at_risk,qualified


In [8]:
# Question 3.1: Naïve baseline model
#The baseline predicts the most frequent class in the training dataset for every municipality. We'll use DummyClassifier and evaluate it using accuracy and macro-F1.
#Step 1: Prepare the dataset
#This separates the input variables (X) from the target (y) and excludes target_opinion_raw to avoid potential data leakage.

# Check the target column
print(df["target_next_year"].value_counts(dropna=False))

# Remove rows where the target is missing
df = df.dropna(subset=["target_next_year"]).copy()

# Remove rows with unexpected target labels, if any
valid_classes = ["at_risk", "not_at_risk"]
df = df[df["target_next_year"].isin(valid_classes)].copy()

# Separate features (X) and target (y)
X = df.drop(columns=["target_next_year", "target_opinion_raw"],
            errors="ignore")
y = df["target_next_year"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget classes:")
print(y.value_counts())


target_next_year
not_at_risk    781
at_risk        653
Name: count, dtype: int64
Features shape: (1434, 14)
Target shape: (1434,)

Target classes:
target_next_year
not_at_risk    781
at_risk        653
Name: count, dtype: int64


In [9]:
#Step 2: Split the data
#We need three separate datasets:
 #Training set: used to train the models
 #Validation set: used to compare the models
 #Test set: reserved for the final evaluation
#This gives us a 70% training, 15% validation and 15% test split. We'll use the same split for all five models.

from sklearn.model_selection import train_test_split

# First split: 70% training, 30% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)


print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)



Training set: (1003, 14)
Validation set: (215, 14)
Test set: (216, 14)


In [10]:
#Step 3: Train the naïve baseline

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Create and train the baseline model
baseline = DummyClassifier(
    strategy="most_frequent",
    random_state=42
)

baseline.fit(X_train, y_train)

# Predict on the validation set
y_val_baseline = baseline.predict(X_val)

# Calculate evaluation metrics
baseline_accuracy = accuracy_score(y_val, y_val_baseline)
baseline_macro_f1 = f1_score(
    y_val,
    y_val_baseline,
    average="macro",
    zero_division=0
)

print("Naive Baseline Results")
print("----------------------")
print("Predicted class:", baseline.classes_[np.argmax(
    np.bincount(baseline.predict(X_train) == baseline.classes_[0])
)] if False else baseline.classes_[0])
print(f"Validation Accuracy: {baseline_accuracy:.4f}")
print(f"Validation Macro-F1: {baseline_macro_f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_baseline,
    labels=["at_risk", "not_at_risk"],
    zero_division=0
))


Naive Baseline Results
----------------------
Predicted class: at_risk
Validation Accuracy: 0.5442
Validation Macro-F1: 0.3524

Classification Report:
              precision    recall  f1-score   support

     at_risk       0.00      0.00      0.00        98
 not_at_risk       0.54      1.00      0.70       117

    accuracy                           0.54       215
   macro avg       0.27      0.50      0.35       215
weighted avg       0.30      0.54      0.38       215



In [11]:
#It checks what the model actually predicts.
print("Actual predictions:")
print(pd.Series(baseline.predict(X_val)).value_counts())

print("\nMajority class learned by baseline:")
print(baseline.predict(X_val)[0])

print("\nCorrect baseline metrics:")
print(f"Accuracy: {accuracy_score(y_val, baseline.predict(X_val)):.4f}")
print(f"Macro-F1: {f1_score(y_val, baseline.predict(X_val), average='macro', zero_division=0):.4f}")


Actual predictions:
not_at_risk    215
Name: count, dtype: int64

Majority class learned by baseline:
not_at_risk

Correct baseline metrics:
Accuracy: 0.5442
Macro-F1: 0.3524


In [12]:
#Question 3.2 Substantive models
#Before training Logistic Regression, Random Forest and the two neural-network models, we need to check the feature types and missing values so we can preprocess them correctly.
#This tells us whether we need to encode text columns, fill missing values, and scale numerical features. These preprocessing steps must be learned from the training set only to avoid data leakage.

print("Feature data types:")
display(X_train.dtypes.to_frame(name="Data type"))

print("\nMissing values in training data:")
display(
    X_train.isnull().sum()
    .sort_values(ascending=False)
    .to_frame(name="Missing values")
)

print("\nNumber of numeric columns:",
      X_train.select_dtypes(include=np.number).shape[1])

print("Number of categorical columns:",
      X_train.select_dtypes(exclude=np.number).shape[1])


Feature data types:


,Data type
demarcation.code,str
financial_year_end.year,int64
total_revenue,float64
total_expenditure,float64
operating_surplus,float64
cash_coverage_ratio,float64
collection_rate,float64
debtor_days,float64
repairs_maintenance_ratio,float64
irregular_expenditure_ratio,float64



Missing values in training data:


,Missing values
collection_rate,83
demarcation.code,0
total_revenue,0
total_expenditure,0
operating_surplus,0
financial_year_end.year,0
cash_coverage_ratio,0
debtor_days,0
repairs_maintenance_ratio,0
irregular_expenditure_ratio,0



Number of numeric columns: 10
Number of categorical columns: 4


In [13]:
#3.2 Substantive models
#Step 1: Build the preprocessing pipeline
#We will fit this preprocessing pipeline separately inside each model's training pipeline, ensuring that imputation, scaling and encoding are learned from training data only.

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Remove identifier columns that should not be used as predictors
excluded_columns = ["Name", "demarcation.code", "target_opinion_raw"]

X_train_model = X_train.drop(columns=excluded_columns, errors="ignore")
X_val_model = X_val.drop(columns=excluded_columns, errors="ignore")
X_test_model = X_test.drop(columns=excluded_columns, errors="ignore")

# Identify numeric and categorical features
numeric_features = X_train_model.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train_model.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# Numeric preprocessing: median imputation and scaling
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing: most frequent imputation and encoding
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

print("\nPreprocessing pipeline created successfully!")


Numeric features: ['financial_year_end.year', 'total_revenue', 'total_expenditure', 'operating_surplus', 'cash_coverage_ratio', 'collection_rate', 'debtor_days', 'repairs_maintenance_ratio', 'irregular_expenditure_ratio', 'grant_dependency']
Categorical features: ['prior_audit_outcome', 'prior_audit_opinion_raw']

Preprocessing pipeline created successfully!


In [14]:
#Step 2: Train Logistic Regression and Random Forest
#This trains both models on exactly the same training data and evaluates them on the same validation data.

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

models = {
    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])
}

results = []

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_model, y_train)

    predictions = model.predict(X_val_model)

    accuracy = accuracy_score(y_val, predictions)
    macro_f1 = f1_score(
        y_val, predictions,
        average="macro",
        zero_division=0
    )

    results.append({
        "Model": name,
        "Validation Accuracy": accuracy,
        "Validation Macro-F1": macro_f1
    })

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro-F1: {macro_f1:.4f}")
    print("\nClassification report:")
    print(classification_report(
        y_val,
        predictions,
        labels=["at_risk", "not_at_risk"],
        zero_division=0
    ))

results_df = pd.DataFrame(results)
display(results_df)



Training Logistic Regression...
Accuracy: 0.8605
Macro-F1: 0.8596

Classification report:
              precision    recall  f1-score   support

     at_risk       0.84      0.86      0.85        98
 not_at_risk       0.88      0.86      0.87       117

    accuracy                           0.86       215
   macro avg       0.86      0.86      0.86       215
weighted avg       0.86      0.86      0.86       215


Training Random Forest...
Accuracy: 0.8651
Macro-F1: 0.8642

Classification report:
              precision    recall  f1-score   support

     at_risk       0.85      0.86      0.85        98
 not_at_risk       0.88      0.87      0.88       117

    accuracy                           0.87       215
   macro avg       0.86      0.86      0.86       215
weighted avg       0.87      0.87      0.87       215



,Model,Validation Accuracy,Validation Macro-F1
0,Logistic Regression,0.860465,0.859587
1,Random Forest,0.865116,0.864164


In [15]:

#check whether PyTorch is available in your Anaconda environment.
try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("PyTorch is ready!")
except ImportError:
    print("PyTorch is not installed.")


PyTorch is not installed.


In [24]:
#Step 1: Install PyTorch

%pip install torch


Note: you may need to restart the kernel to use updated packages.


In [25]:

import torch

print("PyTorch is installed!")
print("Version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())


PyTorch is installed!
Version: 2.14.1+cpu
GPU available: False


In [26]:
#3.2 MLP neural network
#Step 1: Prepare the data for PyTorch
#It uses the preprocessing pipeline we created earlier.

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder

# Encode target labels as 0 and 1
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

print("Class mapping:", dict(zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
)))

# Fit preprocessing on training data only
X_train_processed = preprocessor.fit_transform(X_train_model)
X_val_processed = preprocessor.transform(X_val_model)

# Convert sparse matrices to dense arrays if necessary
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()
if hasattr(X_val_processed, "toarray"):
    X_val_processed = X_val_processed.toarray()

# Convert features to PyTorch tensors
X_train_tensor = torch.tensor(
    X_train_processed, dtype=torch.float32
)
X_val_tensor = torch.tensor(
    X_val_processed, dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_encoded, dtype=torch.long
)
y_val_tensor = torch.tensor(
    y_val_encoded, dtype=torch.long
)

print("Training tensor shape:", X_train_tensor.shape)
print("Validation tensor shape:", X_val_tensor.shape)
print("Data prepared successfully!")


Class mapping: {'at_risk': np.int64(0), 'not_at_risk': np.int64(1)}
Training tensor shape: torch.Size([1003, 18])
Validation tensor shape: torch.Size([215, 18])
Data prepared successfully!


In [27]:
# Step 2: Build and train the MLP
# We'll use two hidden layers with 64 and 32 neurons, ReLU activation, dropout regularisation, and the Adam optimiser. We'll monitor validation loss and stop early if it stops improving.


import copy
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

class MLP(nn.Module):
    def __init__(self, input_size, num_classes=2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.network(x)

mlp = MLP(X_train_tensor.shape[1])

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.001)

best_val_loss = float("inf")
best_state = None
patience = 10
epochs_without_improvement = 0
max_epochs = 100

for epoch in range(max_epochs):
    mlp.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        outputs = mlp(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    mlp.eval()
    with torch.no_grad():
        val_outputs = mlp(X_val_tensor)
        val_loss = criterion(val_outputs, y_val_tensor).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(mlp.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: validation loss = {val_loss:.4f}")

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

mlp.load_state_dict(best_state)
print("MLP training complete!")
print("Best validation loss:", round(best_val_loss, 4))


Epoch 10: validation loss = 0.3861
Epoch 20: validation loss = 0.3719
Early stopping at epoch 28
MLP training complete!
Best validation loss: 0.3688


In [28]:
#Step 3: Evaluate the MLP

from sklearn.metrics import accuracy_score, f1_score, classification_report

mlp.eval()
with torch.no_grad():
    mlp_outputs = mlp(X_val_tensor)
    mlp_predictions_encoded = torch.argmax(
        mlp_outputs, dim=1
    ).cpu().numpy()

mlp_predictions = label_encoder.inverse_transform(
    mlp_predictions_encoded
)

mlp_accuracy = accuracy_score(y_val, mlp_predictions)
mlp_macro_f1 = f1_score(
    y_val, mlp_predictions,
    average="macro",
    zero_division=0
)

print("MLP Validation Accuracy:", round(mlp_accuracy, 4))
print("MLP Validation Macro-F1:", round(mlp_macro_f1, 4))
print("\nClassification report:")
print(classification_report(
    y_val, mlp_predictions,
    labels=["at_risk", "not_at_risk"],
    zero_division=0
))

results.append({
    "Model": "MLP Neural Network",
    "Validation Accuracy": mlp_accuracy,
    "Validation Macro-F1": mlp_macro_f1
})


MLP Validation Accuracy: 0.8605
MLP Validation Macro-F1: 0.8596

Classification report:
              precision    recall  f1-score   support

     at_risk       0.84      0.86      0.85        98
 not_at_risk       0.88      0.86      0.87       117

    accuracy                           0.86       215
   macro avg       0.86      0.86      0.86       215
weighted avg       0.86      0.86      0.86       215



In [29]:
#Next step: Build the second neural network
#Step 1: Train the second neural network
#We'll use a Residual Tabular Neural Network. Unlike the standard MLP, it includes a skip connection that allows information to pass across a hidden layer. This gives us a different neural architecture to compare on your municipal dataset.

# Second architecture: Residual Tabular Neural Network

class ResidualTabularNN(nn.Module):
    def __init__(self, input_size, num_classes=2):
        super().__init__()

        self.input_layer = nn.Linear(input_size, 64)
        self.hidden_layer = nn.Linear(64, 64)
        self.skip_layer = nn.Identity()
        self.output_layer = nn.Linear(64, num_classes)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.relu(self.input_layer(x))

        # Residual block with a skip connection
        residual = self.skip_layer(x)
        hidden = self.relu(self.hidden_layer(x))
        hidden = self.dropout(hidden)
        x = self.relu(hidden + residual)

        return self.output_layer(x)


# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

resnet_model = ResidualTabularNN(
    input_size=X_train_tensor.shape[1]
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    resnet_model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

best_val_loss = float("inf")
best_state = None
patience = 10
epochs_without_improvement = 0
max_epochs = 100

for epoch in range(max_epochs):
    resnet_model.train()

    for xb, yb in train_loader:
        optimizer.zero_grad()
        outputs = resnet_model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    resnet_model.eval()

    with torch.no_grad():
        val_outputs = resnet_model(X_val_tensor)
        val_loss = criterion(
            val_outputs, y_val_tensor
        ).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(
            resnet_model.state_dict()
        )
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1}: "
            f"validation loss = {val_loss:.4f}"
        )

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch + 1}")
        break

resnet_model.load_state_dict(best_state)

print("\nResidual neural network training complete!")
print("Best validation loss:", round(best_val_loss, 4))


Epoch 10: validation loss = 0.3637
Epoch 20: validation loss = 0.3730
Early stopping at epoch 20

Residual neural network training complete!
Best validation loss: 0.3637


In [30]:
#Step 2: Evaluate the model


resnet_model.eval()

with torch.no_grad():
    outputs = resnet_model(X_val_tensor)
    predictions_encoded = torch.argmax(
        outputs, dim=1
    ).cpu().numpy()

resnet_predictions = label_encoder.inverse_transform(
    predictions_encoded
)

resnet_accuracy = accuracy_score(
    y_val, resnet_predictions
)

resnet_macro_f1 = f1_score(
    y_val,
    resnet_predictions,
    average="macro",
    zero_division=0
)

print("Residual Neural Network Results")
print("--------------------------------")
print(f"Validation Accuracy: {resnet_accuracy:.4f}")
print(f"Validation Macro-F1: {resnet_macro_f1:.4f}")

print("\nClassification report:")
print(classification_report(
    y_val,
    resnet_predictions,
    labels=["at_risk", "not_at_risk"],
    zero_division=0
))

results.append({
    "Model": "Residual Tabular Neural Network",
    "Validation Accuracy": resnet_accuracy,
    "Validation Macro-F1": resnet_macro_f1
})


Residual Neural Network Results
--------------------------------
Validation Accuracy: 0.8605
Validation Macro-F1: 0.8596

Classification report:
              precision    recall  f1-score   support

     at_risk       0.84      0.86      0.85        98
 not_at_risk       0.88      0.86      0.87       117

    accuracy                           0.86       215
   macro avg       0.86      0.86      0.86       215
weighted avg       0.86      0.86      0.86       215



In [31]:
#Step 3: Display your model comparison
#This table should contain four substantive models: Logistic Regression, Random Forest, MLP, and the Residual Tabular Neural Network.
display(
    pd.DataFrame(results).sort_values(
        by="Validation Macro-F1",
        ascending=False
    ).reset_index(drop=True)
)


,Model,Validation Accuracy,Validation Macro-F1
0,Random Forest,0.865116,0.864164
1,Logistic Regression,0.860465,0.859587
2,MLP Neural Network,0.860465,0.859587
3,Residual Tabular Neural Network,0.860465,0.859587


In [32]:
#Correct the baseline display
print("Actual baseline predictions:")
print(pd.Series(baseline.predict(X_val)).value_counts())

print("\nBaseline validation metrics:")
print(f"Accuracy: {accuracy_score(y_val, baseline.predict(X_val)):.4f}")
print(f"Macro-F1: {f1_score(y_val, baseline.predict(X_val), average='macro', zero_division=0):.4f}")


Actual baseline predictions:
not_at_risk    215
Name: count, dtype: int64

Baseline validation metrics:
Accuracy: 0.5442
Macro-F1: 0.3524


In [33]:
#3. Final evaluation on the test set
#We selected Random Forest using the validation results. Now we'll evaluate it on the untouched test set. This gives us an estimate of how well it generalises to unseen municipalities.
#This does not retrain the model or change its settings; it evaluates the existing models on the test set.

# Final evaluation: selected champion and baseline
rf_test_predictions = models["Random Forest"].predict(X_test_model)
baseline_test_predictions = baseline.predict(X_test)

print("RANDOM FOREST — FINAL TEST RESULTS")
print("----------------------------------")
print(f"Accuracy: {accuracy_score(y_test, rf_test_predictions):.4f}")
print(f"Macro-F1: {f1_score(y_test, rf_test_predictions, average='macro', zero_division=0):.4f}")
print("\nClassification report:")
print(classification_report(
    y_test,
    rf_test_predictions,
    labels=["at_risk", "not_at_risk"],
    zero_division=0
))

print("\nNAÏVE BASELINE — TEST RESULTS")
print("-----------------------------")
print(f"Accuracy: {accuracy_score(y_test, baseline_test_predictions):.4f}")
print(f"Macro-F1: {f1_score(y_test, baseline_test_predictions, average='macro', zero_division=0):.4f}")



RANDOM FOREST — FINAL TEST RESULTS
----------------------------------
Accuracy: 0.8472
Macro-F1: 0.8468

Classification report:
              precision    recall  f1-score   support

     at_risk       0.80      0.88      0.84        98
 not_at_risk       0.89      0.82      0.85       118

    accuracy                           0.85       216
   macro avg       0.85      0.85      0.85       216
weighted avg       0.85      0.85      0.85       216


NAÏVE BASELINE — TEST RESULTS
-----------------------------
Accuracy: 0.5463
Macro-F1: 0.3533
